In [1]:
import os
import sys

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.text_utils import find_index_of_incongruent_word
from src.tokenizer_utils import load_tokenizer, tokenize

### Representation Similarity Analysis

Задача вычислить пространство представлений вычислением расстояний между стимулами в пространстве признаков

- признаковое простраство для модели - скрытые представления стимулов на каждом из слоев
- признаковое пространство для ЭЭГ - временные ряды значений потенциалов с датчиков

In [3]:
result_csv_filename = "results.csv"

non_instruct_paths = [
    "../src/results/meta-llama_Meta-Llama-3-8B/",
    "../src/results/mistralai_Mistral-7B-v0.1/",
    "../src/results/Qwen_Qwen2.5-7B/",
]

In [4]:
test_path = non_instruct_paths[1]

df_result = pd.read_csv(os.path.join(test_path, result_csv_filename))

In [9]:
test_path

'../src/results/mistralai_Mistral-7B-v0.1/'

In [5]:
df_result.head()

,sentence,model,instruct,hidden_states_path,structure,target,congruent
0,Автобусы проходят массовую дезинфекцией,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00000_mist...,Subject - Verb - Adj - Object,grammar,Автобусы проходят массовую дезинфекцию
1,Автобусы проходят массовую дезинфекцию,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00001_mist...,Subject - Verb - Adj - Object,normal,Автобусы проходят массовую дезинфекцию
2,Автобусы проходят массовую фортуной,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00002_mist...,Subject - Verb - Adj - Object,semantics_grammar,Автобусы проходят массовую дезинфекцию
3,Автобусы проходят массовую фортуну,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00003_mist...,Subject - Verb - Adj - Object,semantics,Автобусы проходят массовую дезинфекцию
4,Авторы получали подарками,mistralai/Mistral-7B-v0.1,False,./results/mistralai_Mistral-7B-v0.1/00004_mist...,Subject - Verb - Object,grammar,Авторы получали подарки


In [6]:

row_1 = df_result.iloc[0]

row_1

sentence                        Автобусы проходят массовую дезинфекцией
model                                         mistralai/Mistral-7B-v0.1
instruct                                                          False
hidden_states_path    ./results/mistralai_Mistral-7B-v0.1/00000_mist...
structure                                 Subject - Verb - Adj - Object
target                                                          grammar
congruent                        Автобусы проходят массовую дезинфекцию
Name: 0, dtype: object

In [7]:
find_index_of_incongruent_word([row_1["sentence"]], [row_1["congruent"]])


tokenizer = load_tokenizer(row_1["model"])

encoded_sentence = tokenize(
    tokenizer=tokenizer,
    sentences=[row_1["sentence"]],
    use_chat_template=row_1["instruct"],
)

2025-10-31 01:32:49 DEBUG    src.text_utils: Searching for incongruent word: дезинфекцию
2025-10-31 01:32:49 DEBUG    urllib3.connectionpool: Starting new HTTPS connection (1): huggingface.co:443
2025-10-31 01:32:49 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /mistralai/Mistral-7B-v0.1/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
2025-10-31 01:32:49 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/mistralai/Mistral-7B-v0.1/27d67f1b5f57dc0953326b2601d68371d40ea8da/tokenizer_config.json HTTP/1.1" 200 0
2025-10-31 01:32:49 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "GET /api/models/mistralai/Mistral-7B-v0.1/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
2025-10-31 01:32:49 INFO     src.tokenizer_utils: Tokenizer adds prefix space


In [8]:
for file in os.listdir(test_path):
    if ".npy" in file:
        hiddens = np.load(os.path.join(test_path, file))
        print(hiddens.shape)
        break

(20, 33, 4096)


In [18]:
import pathlib

embeddings = []

for i, row in df_result.iterrows():
    hidden_state_path = pathlib.Path(test_path) / row["hidden_states_path"].split("/")[-1]
    hs = np.load(hidden_state_path)
    embeddings.append(hs[-1, -1, :])

In [21]:
np.stack(embeddings)

array([[-1.8874105 , -2.5114436 , -1.7923547 , ...,  0.56642383,
         1.7378137 ,  3.374642  ],
       [ 0.26383016, -0.9291273 , -2.2444844 , ...,  1.3234282 ,
         2.0872033 ,  3.2661638 ],
       [-3.8437123 , -0.20322703, -0.10747155, ...,  1.0838323 ,
        -0.04078794,  1.0846466 ],
       ...,
       [ 2.701474  ,  2.2553914 ,  1.8662494 , ...,  1.4528245 ,
        -0.32555774,  2.5376146 ],
       [-0.6781415 , -0.9743209 ,  2.915815  , ...,  5.8672256 ,
         0.6943762 , -1.511294  ],
       [-3.9519742 , -5.7168593 ,  2.8273444 , ...,  2.7121387 ,
        -2.3100266 ,  0.03342525]], shape=(600, 4096), dtype=float32)

In [22]:
from src.rsa_v2 import rsa_v2

In [23]:
rsa_v2(np.stack(embeddings), np.stack(embeddings))

IndexError: index 600 is out of bounds for axis 1 with size 600

In [28]:
s = np.stack(embeddings)

In [29]:
import numpy as np
from scipy.stats import zscore, spearmanr
from scipy.spatial.distance import pdist, squareform

In [33]:
X_norm = zscore(s, axis=0, ddof=1)
condensed = pdist(X_norm, metric="cosine")
X_rdm = squareform(condensed)

In [38]:
idx = np.triu_indices_from(X_rdm, k=1)

In [39]:
idx

(array([  0,   0,   0, ..., 597, 597, 598], shape=(179700,)),
 array([  1,   2,   3, ..., 598, 599, 599], shape=(179700,)))

In [40]:
X_rdm[idx]

array([0.04539116, 0.84437184, 0.81318091, ..., 1.03149859, 0.88410243,
       0.35702706], shape=(179700,))

In [41]:
from datasets import load_dataset

2025-10-31 02:34:51 DEBUG    datasets: PyTorch version 2.8.0 available.


In [43]:
ds = load_dataset("ContributorsSIGNAL/SIGNAL")

2025-10-31 02:35:25 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /datasets/ContributorsSIGNAL/SIGNAL/resolve/main/README.md HTTP/1.1" 307 0
2025-10-31 02:35:25 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/datasets/ContributorsSIGNAL/SIGNAL/a21068568606a9b51a27e23ea3edbc78b5b6478c/README.md HTTP/1.1" 200 0
2025-10-31 02:35:25 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /datasets/ContributorsSIGNAL/SIGNAL/resolve/a21068568606a9b51a27e23ea3edbc78b5b6478c/SIGNAL.py HTTP/1.1" 404 0
2025-10-31 02:35:25 DEBUG    urllib3.connectionpool: https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/datasets/ContributorsSIGNAL/SIGNAL/ContributorsSIGNAL/SIGNAL.py HTTP/1.1" 404 0
2025-10-31 02:35:26 DEBUG    urllib3.connectionpool: https://huggingface.co:443 "HEAD /datasets/ContributorsSIGNAL/SIGNAL/resolve/a21068568606a9b51a27e23ea3edbc78b5b6478c/.huggingface.yaml HTTP/1.1" 404 0
2025-10-31 02:35:26 DEBUG   

In [45]:
ds["train"]

Dataset({
    features: ['sentence_id', 'sentence', 'congruent', 'structure', 'length', 'target', 'position', 'most_popular', 'percent', 'semantics_grammar', 'semantics', 'grammar', 'no', 'unknown', 'subject', 'verb', 'object', 'gen', 'adj', 'subject_lemma', 'subject_length', 'subject_gender', 'subject_ipm', 'verb_lemma', 'verb_length', 'verb_ipm', 'object_lemma', 'object_length', 'object_gender', 'object_ipm', 'gen_lemma', 'gen_length', 'gen_gender', 'gen_ipm', 'adj_lemma', 'adj_length', 'adj_gender', 'adj_ipm'],
    num_rows: 600
})

In [97]:
from pathlib import Path

dataset_local_path = Path("../hf_datasets")

epochs_info = dataset_local_path / "epochs_info" / "p9_events.csv"

In [98]:
import pandas as pd

In [99]:
pd.read_csv(epochs_info)

,sentence_id,sentence_id_eeg,sentence,structure,position,target,event_start,event_name
0,295,140,Суд принял соответствующее путешествие,Subject - Verb - Adj - Object,3,semantics,18258,Stimulus/S 2_2
1,201,95,Полиция разыскивает владелец здания,Subject - Verb - Object - Gen,2,grammar,21757,Stimulus/S 3_3
2,326,150,Цены показывают положительную фамилию,Subject - Verb - Adj - Object,3,semantics,25274,Stimulus/S 2_2
3,331,152,Шведы потерпели поражения,Subject - Verb - Object,2,grammar,28789,Stimulus/S 3_1
4,129,65,Незнакомцы затеяли ссору,Subject - Verb - Object,2,normal,31806,Stimulus/S 1_1
...,...,...,...,...,...,...,...,...
595,281,130,Спецназ продолжает удобству бандитов,Subject - Verb - Object - Gen,2,semantics_grammar,2447785,Stimulus/S 4_3
596,117,61,Милиция провела проверке,Subject - Verb - Object,2,grammar,2451301,Stimulus/S 3_1
597,336,154,Эксперты сделали вывода,Subject - Verb - Object,2,grammar,2454318,Stimulus/S 3_1
598,203,97,Пользователи обратили село,Subject - Verb - Object,2,semantics,2457335,Stimulus/S 2_1
